In [2]:
import pandas as pd
import numpy as np
import os
import gc
from sklearn.model_selection import GroupShuffleSplit
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize

# 1. Configuración de rutas y columnas
dir_entrada = "../../Datos/Datos procesados"
columnas_a_cargar = [
    'CIP_ENCRIPTADO', 'MORTALIDAD', 'SEVERIDAD', 'CONSUMO_RECURSOS', 
    'CATEGORIA_CANCER', 'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 
    'ES_PUEBLO_ORIGINARIO', 'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 
    'REGION', 'SERVICIOINGRESO', 'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 
    'TIPO_INGRESO', 'TIPO_PREVISION', 'TIPO_PROCEDENCIA', 'CANTIDAD_TRASLADOS', 
    'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS'
]

vars_para_ohe = [
    'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 'ES_PUEBLO_ORIGINARIO', 
    'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 'REGION', 'SERVICIOINGRESO', 
    'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 'TIPO_INGRESO', 'TIPO_PREVISION', 
    'TIPO_PROCEDENCIA', 'CATEGORIA_CANCER' 
]

print("=== INICIANDO PRUEBA DE ROBUSTEZ FINAL (3 TARGETS) ===")

lista_df_onco = []

# 2. Cargar todos los años y filtrar solo Oncológicos
for año in range(2019, 2025):
    archivo = f"GRD_PROCESADO_{año}_DERIVADAS.csv"
    ruta = os.path.join(dir_entrada, archivo)
    
    if os.path.exists(ruta):
        print(f"Cargando {año}...")
        df_temp = pd.read_csv(ruta, usecols=columnas_a_cargar, low_memory=False)
        
        df_temp['CATEGORIA_CANCER'] = df_temp['CATEGORIA_CANCER'].astype(str).str.split(':').str[0].str.replace('-', '_').str.strip()
        df_onco_temp = df_temp[~df_temp['CATEGORIA_CANCER'].str.contains('SIN_CANCER', na=False)].copy()
        
        lista_df_onco.append(df_onco_temp)
        del df_temp
        gc.collect()

# Unificar dataset
df_onco_global = pd.concat(lista_df_onco, ignore_index=True)
del lista_df_onco
gc.collect()

# 3. LIMPIEZA DE IDs CORRUPTOS
print("\nLimpiando IDs corruptos...")
df_onco_global = df_onco_global.dropna(subset=['CIP_ENCRIPTADO'])
df_onco_global['CIP_ENCRIPTADO'] = df_onco_global['CIP_ENCRIPTADO'].astype(str).str.strip()
ids_corruptos = ['SIN INFORMACIÓN', 'SIN INFORMACIÃ“N', '95162030', '95162030.0', '78492052', '78492052.0', 'nan', 'NAN']
df_onco_global = df_onco_global[~df_onco_global['CIP_ENCRIPTADO'].isin(ids_corruptos)]

# 4. PREPARAR MATRIZ X Y VECTORES Y
print("Aplicando One-Hot Encoding...")
df_onco_ohe = pd.get_dummies(df_onco_global.drop(columns=['CIP_ENCRIPTADO']), columns=vars_para_ohe, drop_first=True)
df_onco_ohe.columns = df_onco_ohe.columns.str.replace(' ', '_').str.replace('-', '_').str.upper()

# Matriz de características (sin ningún target)
X = df_onco_ohe.drop(columns=['MORTALIDAD', 'SEVERIDAD', 'CONSUMO_RECURSOS'])
groups = df_onco_global['CIP_ENCRIPTADO']

print(f"Total episodios limpios: {len(X):,}")
print(f"Total pacientes únicos: {groups.nunique():,}")

# 5. SPLIT ÚNICO AGRUPADO POR PACIENTE
print("\nRealizando partición GroupShuffleSplit...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

del df_onco_global
gc.collect()

=== INICIANDO PRUEBA DE ROBUSTEZ FINAL (3 TARGETS) ===
Cargando 2019...
Cargando 2020...
Cargando 2021...
Cargando 2022...
Cargando 2023...
Cargando 2024...

Limpiando IDs corruptos...
Aplicando One-Hot Encoding...
Total episodios limpios: 493,154
Total pacientes únicos: 284,547

Realizando partición GroupShuffleSplit...


0

In [3]:
# Función auxiliar para calcular métricas multiclase y binarias de forma elegante
def evaluar_metricas(y_true, y_pred, y_prob, es_multiclase=False):
    f1_macro = f1_score(y_true, y_pred, average='macro')
    
    if not es_multiclase:
        f1_clase1 = f1_score(y_true, y_pred, pos_label=1) # <-- AQUÍ AGREGAMOS LA CLASE 1
        auc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
        return f1_macro, f1_clase1, auc, auprc
    else:
        # AUC multiclase
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
        # Binarizar el y_true para calcular AUPRC ponderado (One-vs-Rest)
        clases = np.unique(y_true)
        y_true_bin = label_binarize(y_true, classes=clases)
        auprc = average_precision_score(y_true_bin, y_prob, average='weighted')
        return f1_macro, None, auc, auprc

print("\n" + "="*50)
print("ENTRENAMIENTO Y EVALUACIÓN POR TARGET")
print("="*50)

# ==========================================
# TARGET 1: MORTALIDAD (BINARIO) -> Random Forest
# ==========================================
print("\n1. Entrenando MORTALIDAD (Random Forest)...")
y_mort = df_onco_ohe['MORTALIDAD']
y_train_m, y_test_m = y_mort.iloc[train_idx], y_mort.iloc[test_idx]

modelo_rf = RandomForestClassifier(
    n_estimators=500, max_depth=35, min_samples_split=10, 
    class_weight='balanced', n_jobs=-1, random_state=42
)
modelo_rf.fit(X_train, y_train_m)
y_pred_m = modelo_rf.predict(X_test)
y_prob_m = modelo_rf.predict_proba(X_test)[:, 1]

f1_m_macro, f1_m_c1, auc_m, auprc_m = evaluar_metricas(y_test_m, y_pred_m, y_prob_m, es_multiclase=False)
print(f"-> F1-Macro: {f1_m_macro:.4f} | F1-Clase 1: {f1_m_c1:.4f} | AUC-ROC: {auc_m:.4f} | AUPRC: {auprc_m:.4f}")

# ==========================================
# TARGET 2: SEVERIDAD (MULTICLASE) -> XGBoost
# ==========================================
print("\n2. Entrenando SEVERIDAD (XGBoost Multiclase)...")
y_sev = df_onco_ohe['SEVERIDAD']
y_train_s, y_test_s = y_sev.iloc[train_idx], y_sev.iloc[test_idx]

modelo_xgb_sev = xgb.XGBClassifier(
    learning_rate=0.3, max_depth=10, tree_method='hist', 
    n_jobs=-1, random_state=42
)
modelo_xgb_sev.fit(X_train, y_train_s)
y_pred_s = modelo_xgb_sev.predict(X_test)
y_prob_s = modelo_xgb_sev.predict_proba(X_test)

f1_s_macro, _, auc_s, auprc_s = evaluar_metricas(y_test_s, y_pred_s, y_prob_s, es_multiclase=True)
print(f"-> F1-Macro: {f1_s_macro:.4f} | AUC-ROC: {auc_s:.4f} | AUPRC: {auprc_s:.4f}")

# ==========================================
# TARGET 3: CONSUMO DE RECURSOS (MULTICLASE) -> XGBoost
# ==========================================
print("\n3. Entrenando CONSUMO DE RECURSOS (XGBoost Multiclase)...")
y_cons = df_onco_ohe['CONSUMO_RECURSOS']
y_train_c, y_test_c = y_cons.iloc[train_idx], y_cons.iloc[test_idx]

modelo_xgb_cons = xgb.XGBClassifier(
    learning_rate=0.3, max_depth=10, tree_method='hist', 
    n_jobs=-1, random_state=42
)
modelo_xgb_cons.fit(X_train, y_train_c)
y_pred_c = modelo_xgb_cons.predict(X_test)
y_prob_c = modelo_xgb_cons.predict_proba(X_test)

f1_c_macro, _, auc_c, auprc_c = evaluar_metricas(y_test_c, y_pred_c, y_prob_c, es_multiclase=True)
print(f"-> F1-Macro: {f1_c_macro:.4f} | AUC-ROC: {auc_c:.4f} | AUPRC: {auprc_c:.4f}")

print("\n=== FIN DE LA PRUEBA DE ROBUSTEZ ===")


ENTRENAMIENTO Y EVALUACIÓN POR TARGET

1. Entrenando MORTALIDAD (Random Forest)...
-> F1-Macro: 0.6896 | F1-Clase 1: 0.4394 | AUC-ROC: 0.9231 | AUPRC: 0.4342

2. Entrenando SEVERIDAD (XGBoost Multiclase)...
-> F1-Macro: 0.7712 | AUC-ROC: 0.9066 | AUPRC: 0.8145

3. Entrenando CONSUMO DE RECURSOS (XGBoost Multiclase)...
-> F1-Macro: 0.7431 | AUC-ROC: 0.8950 | AUPRC: 0.8635

=== FIN DE LA PRUEBA DE ROBUSTEZ ===
